# ASII News Sentiment (Sectors + OpenAI)

This notebook:
1. Pulls recent news headlines mentioning **Astra International (ASII.JK)** from the [Sectors API](https://sectors.app)'s news endpoint.
2. Classifies each headline's sentiment (positive / negative) using the OpenAI (ChatGPT) API.
3. Saves the labeled headlines to CSV and shows a quick summary.

**Before running:**
- Get a Sectors API key at https://sectors.app (the news endpoint costs 1 API credit per request, up to 30 articles per call).
- Get an OpenAI API key at https://platform.openai.com/api-keys
- Either set them as environment variables (`SECTORS_API_KEY`, `OPENAI_API_KEY`) before launching Jupyter, or just run the "get API keys" cell below and paste them in when prompted -- they're kept in memory only for this session, never written into this notebook file.

In [ ]:
%pip install -q requests openai pandas matplotlib

In [1]:
import os
import time
import getpass
import requests
import pandas as pd
import matplotlib.pyplot as plt
from openai import OpenAI
from datetime import datetime


In [2]:
# Keys live in ../.env (gitignored); never paste a real key into a cell.
import sys
sys.path.append("../scripts")
from load_env import load_env
load_env()

SECTORS_API_KEY = os.environ.get("SECTORS_API_KEY") or getpass.getpass("Sectors API key: ")
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY") or getpass.getpass("OpenAI API key: ")

client = OpenAI(api_key=OPENAI_API_KEY)


# Step 1: Pull News Headlines from Sectors

Sectors' `GET /v2/news/` endpoint returns general IDX news, filterable by `symbols` (ticker) or a free-text `keyword`. We page through results with `limit`/`offset` until we've collected as many as we want or run out of pages.

Note: in Sectors' own sample responses, many articles carry an **empty `symbols` list** even when they're clearly about a specific company -- ticker tagging looks sparse. So we pull by ticker symbol *and* by keyword ("Astra International") and merge the two, rather than trusting the ticker filter alone.

In [3]:
SECTORS_BASE_URL = "https://api.sectors.app/v2/news/"
TICKER = "ASII"          # Sectors uses the bare ticker, no ".JK" suffix
COMPANY_KEYWORD = "Astra International"

def fetch_news(symbols=None, keyword=None, start="2025-09-01", end=None, limit=30, max_articles=300):
    """
    Pull news articles from the Sectors /v2/news/ endpoint, paging through
    results until either `max_articles` is reached or the API reports no
    more pages. Pass `symbols` (ticker) and/or `keyword` (free-text search).
    """
    # end defaults to today, not a fixed past date -- otherwise every call
    # that doesn't override `end` (like the Step 1 "recent news" pull below)
    # silently misses everything newer than whatever date was last hardcoded here.
    if end is None:
        end = datetime.today().strftime("%Y-%m-%d")
    headers = {"Authorization": SECTORS_API_KEY}
    articles = []
    offset = 0

    while len(articles) < max_articles:
        # params=, not manual f-string concatenation -- requests handles
        # URL-encoding for us (COMPANY_KEYWORD has a space in it, which a
        # raw f-string URL would send un-encoded and likely mangle).
        params = {"limit": limit, "extension": "idx", "start": start, "end": end, "offset": offset}
        if symbols:
            params["symbols"] = symbols
        if keyword:
            params["keyword"] = keyword

        resp = requests.get(SECTORS_BASE_URL, headers=headers, params=params)
        resp.raise_for_status()
        data = resp.json()
        articles.extend(data["results"])
        if not data["pagination"]["has_next"]:
            break
        offset += limit

    return articles[:max_articles]


In [4]:
by_symbol = fetch_news(symbols=TICKER, max_articles=100)
by_keyword = fetch_news(keyword=COMPANY_KEYWORD, max_articles=100)

print(f"By ticker symbol ({TICKER}): {len(by_symbol)} articles")
print(f"By keyword ({COMPANY_KEYWORD!r}): {len(by_keyword)} articles")

news_df = pd.concat([pd.DataFrame(by_symbol), pd.DataFrame(by_keyword)], ignore_index=True)

if not news_df.empty:
    news_df = news_df.drop(columns=["body"], errors="ignore")  # not needed downstream, and it's the bulk of each row
    news_df = news_df.drop_duplicates(subset="title").reset_index(drop=True)
    # format="mixed": the API returns ISO timestamps like "...T23:59:59",
    # but a CSV round-trip through pandas re-stringifies them as
    # "... 23:59:59" (space, not "T") -- once these get combined (in the
    # backfill step below) the column has both styles, which breaks a
    # fixed-format parse. "mixed" infers per-element instead of assuming
    # one format for the whole column.
    news_df["timestamp"] = pd.to_datetime(news_df["timestamp"], format="mixed")
    news_df = news_df.sort_values("timestamp").reset_index(drop=True)

print(f"\n{len(news_df)} unique articles after merging + de-duplicating by title")
news_df[["timestamp", "title", "source"]].tail(10) if not news_df.empty else news_df

By ticker symbol (ASII): 100 articles
By keyword ('Astra International'): 100 articles

152 unique articles after merging + de-duplicating by title


,timestamp,title,source
142,2026-09-23 18:21:00,PT Astra International Tbk (ASII) plans major ...,https://investasi.kontan.co.id/news/asii-siapk...
143,2026-09-23 19:01:00,PT Astra International Tbk (ASII) Tightens Inv...,https://emitennews.com/news/di-luar-3-bisnis-i...
144,2026-09-23 20:55:00,"ASII shares slip below IDR5,000 as President C...",https://www.idnfinancials.com/id/news/69326/ha...
145,2026-09-24 05:40:00,MNC Sekuritas forecasts IHSG rise and recommen...,https://market.bisnis.com/read/20260924/7/2006...
146,2026-09-24 06:20:43,President Director Rudi and Commissioner Prijo...,https://www.cnbcindonesia.com/market/202609240...
147,2026-09-24 08:59:00,Two PT Astra International Tbk (ASII) executiv...,https://market.bisnis.com/read/20260924/7/2006...
148,2026-09-24 22:07:00,"Foreign investors ramp up net selling, surpass...",https://investor.id/market/455426/asing-meraja...
149,2026-09-25 14:25:24,Thomas Junaidi Alim W buy Shares of Astra Inte...,https://www.idx.co.id/StaticData/NewsAndAnnoun...
150,2026-09-25 15:14:33,Djap Tet Fa Buys Shares of Astra International,https://www.idx.co.id/StaticData/NewsAndAnnoun...
151,2026-09-25 17:16:00,Directors and Commissioner of PT Astra Interna...,https://emitennews.com/news/saham-astra-intern...


In [5]:
news_df.head()

,title,source,thumbnail,timestamp,sector,sub_sector,tags,symbols,dimension
0,Astra International's May 2026 Car Sales Rise ...,https://money.kompas.com/read/2026/06/11/16083...,https://asset.kompas.com/crops/vhEwoRUr6Sx224z...,2026-06-11 16:08:00,industrials,[multi-sector-holdings],"[Business Expansion, Financial Metrics, Produc...",[ASII.JK],"{'future': 0, 'dividend': 0, 'ownership': 0, '..."
1,PT Astra International Tbk reports May 2026 ve...,https://money.kompas.com/read/2026/06/11/16175...,https://asset.kompas.com/crops/K8nX5SVuJZYbUpJ...,2026-06-11 16:17:00,industrials,[multi-sector-holdings],"[Financial Metrics, Business Expansion, Produc...",[ASII.JK],"{'future': 0, 'dividend': 0, 'ownership': 0, '..."
2,Indonesian issuers launch jumbo share buybacks...,https://investasi.kontan.co.id/news/sederet-em...,https://foto.kontan.co.id/sTgZzZOBW_v0SnrugD3X...,2026-06-16 18:47:00,industrials,"[multi-sector-holdings, telecommunication, ban...","[Stock Buyback, Analyst Ratings, Capital & Fun...","[ASII.JK, TLKM.JK, BBRI.JK, NCKL.JK, PANS.JK]","{'future': 0, 'dividend': 0, 'ownership': 0, '..."
3,PT Astra International Tbk announces Rp 8 tril...,https://investasi.kontan.co.id/news/buyback-ju...,https://foto.kontan.co.id/nkU32RLj1wR9cRFeuw8F...,2026-06-17 07:57:00,industrials,"[multi-sector-holdings, telecommunication, ban...","[Stock Buyback, Analyst Ratings, Market Sentim...","[ASII.JK, TLKM.JK, BBRI.JK, NCKL.JK, PANS.JK]","{'future': 0, 'dividend': 0, 'ownership': 0, '..."
4,Astra International completes Rp810.7 billion ...,https://emitennews.com/news/baru-tuntas-asii-l...,https://ap-south-1.linodeobjects.com/emitennew...,2026-06-17 08:35:00,industrials,[multi-sector-holdings],"[Stock Buyback, Shareholders General Meeting, ...",[ASII.JK],"{'future': 2, 'dividend': 0, 'ownership': 0, '..."


## Backfill Earlier History

The pull above only reaches back to whatever `max_articles` happened to cover -- in practice, the last few months. To get a daily sentiment feature that actually has something to train on across the *whole* price history in `data/prices/ASII.csv` (which starts 2025-09-26), we need news from further back too.

This backfills **2025-09-01 to 2026-05-30** (the earliest date the first pull already reached) and combines it with whatever was already pulled and saved to `data/news/ASII/asii_news_raw.csv`, instead of re-pulling that recent window again.

**Cost note:** each Sectors request returns up to 30 articles and costs 1 API credit. A ~9-month window can need many pages -- `max_articles` below caps the spend; raise/lower it deliberately.

In [ ]:
BACKFILL_START = "2025-09-01"
BACKFILL_END = "2026-05-30"  # earliest date the first pull already reached

print(f"Pulling backfill articles ({BACKFILL_START} to {BACKFILL_END})...")
by_symbol_backfill = fetch_news(symbols=TICKER, start=BACKFILL_START, end=BACKFILL_END, max_articles=900)
by_keyword_backfill = fetch_news(keyword=COMPANY_KEYWORD, start=BACKFILL_START, end=BACKFILL_END, max_articles=900)

backfill_df = pd.concat([pd.DataFrame(by_symbol_backfill), pd.DataFrame(by_keyword_backfill)], ignore_index=True)
print(f"Backfill: {len(by_symbol_backfill)} by symbol, {len(by_keyword_backfill)} by keyword -> {len(backfill_df)} rows before dedup")

# Combine with whatever was already pulled (and possibly already
# classified) rather than re-spending API credits re-pulling the recent
# window again. Falls back to the in-memory `news_df` from the cell above
# only if nothing has been saved to disk yet.
existing_raw_path = "../data/news/ASII/asii_news_raw.csv"
if os.path.exists(existing_raw_path):
    existing_df = pd.read_csv(existing_raw_path)
    print(f"Loaded {len(existing_df)} previously pulled articles from {existing_raw_path} (not re-pulling that range)")
else:
    existing_df = news_df.copy()

news_df = pd.concat([existing_df, backfill_df], ignore_index=True)
news_df = news_df.drop(columns=["body"], errors="ignore")  # drop any leftover body column from older pulls too
news_df = news_df.drop_duplicates(subset="title").reset_index(drop=True)
# format="mixed": existing_df's timestamps came back from a previous CSV
# round-trip ("... 23:59:59", space-separated), while backfill_df's are
# fresh from the API ("...T23:59:59", ISO with "T") -- the column now has
# both styles, so a fixed-format parse breaks. "mixed" infers per-element.
news_df["timestamp"] = pd.to_datetime(news_df["timestamp"], format="mixed")
news_df = news_df.sort_values("timestamp").reset_index(drop=True)

print(f"\nCombined total: {len(news_df)} unique articles, {news_df['timestamp'].min().date()} to {news_df['timestamp'].max().date()}")
news_df[["timestamp", "title", "source"]].head(10)

## Diagnostic: is this ASII-specific, or does the archive not reach back that far at all?

The backfill above found almost nothing in `2025-09-01` to `2026-05-30`. Before concluding "Sectors has no ASII history that far back," check whether the API has *any* IDX news at all in that window -- no ticker/keyword filter, just the date range.

In [5]:
diag = fetch_news(start="2025-09-01", end="2026-05-30", max_articles=30)
print(f"Any general IDX news in that window (no ticker/keyword filter): {len(diag)}")
if diag:
    print(pd.DataFrame(diag)[["timestamp", "title"]].head(5))

Any general IDX news in that window (no ticker/keyword filter): 12
             timestamp                                              title
0  2026-05-30T23:59:59  DEWA, UNTR, and UNVR highlighted as key stocks...
1  2026-05-30T23:59:59  PT ITSEC Asia Tbk posts 62% revenue growth and...
2  2026-05-30T20:59:12  GOTO risks MSCI index removal and FCA board tr...
3  2026-05-30T20:48:00  PT Kereta Cepat Indonesia China reports 15% ri...
4  2026-05-30T18:13:00  Harita Nickel reports Q1 2026 revenue of Rp 6....


**If `news_df` came back empty or very small:** double check the `TICKER`/`COMPANY_KEYWORD` spelling, widen the `start`/`end` date range (default has no lower bound, but worth checking), and confirm your Sectors API key has news access on your plan.

In [ ]:
os.makedirs("../data/news/ASII", exist_ok=True)
news_df.to_csv("../data/news/ASII/asii_news_raw.csv", index=False)
print(f"Saved {len(news_df)} articles to ../data/news/ASII/asii_news_raw.csv")

In [ ]:
SENTIMENT_MODEL = "gpt-4o-mini"  # cheap + fast; swap for whichever chat model you have access to

SENTIMENT_SYSTEM_PROMPT = (
    "You are a financial news sentiment classifier. Given a single news "
    "headline about a company, respond with exactly one word: "
    "'positive' if the headline is good news for the company/its stock, "
    "or 'negative' if it's bad news. If it reads as neutral or purely "
    "factual, still make your best-judgment call rather than refusing -- "
    "always answer with exactly one of those two words, nothing else."
)

def classify_sentiment(headline, retries=3):
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=SENTIMENT_MODEL,
                messages=[
                    {"role": "system", "content": SENTIMENT_SYSTEM_PROMPT},
                    {"role": "user", "content": headline},
                ],
                temperature=0,
                max_tokens=3,
            )
            label = resp.choices[0].message.content.strip().lower()
            if label in ("positive", "negative"):
                return label
            # Model didn't follow the format exactly -- take the best guess
            # rather than silently dropping the row.
            if "pos" in label:
                return "positive"
            if "neg" in label:
                return "negative"
            return "unclear"
        except Exception as e:
            if attempt == retries - 1:
                print(f"  Failed after {retries} attempts on {headline!r}: {e}")
                return "error"
            time.sleep(2 ** attempt)  # backoff: 1s, 2s, 4s

In [ ]:
# Reuse sentiment labels already saved from a previous run -- only spend
# OpenAI calls on genuinely new headlines (i.e. the backfill).
existing_sentiment_path = "../data/news/ASII/asii_news_sentiment.csv"
if os.path.exists(existing_sentiment_path):
    labeled_lookup = pd.read_csv(existing_sentiment_path)[["title", "sentiment"]].drop_duplicates("title")
    news_df = news_df.merge(labeled_lookup, on="title", how="left")
else:
    news_df["sentiment"] = pd.NA

to_classify = news_df["sentiment"].isna()
print(f"{(~to_classify).sum()} headlines already labeled previously (reused, no API cost)")
print(f"{to_classify.sum()} new headlines to classify now")

todo_idx = news_df.index[to_classify]
for n, i in enumerate(todo_idx):
    news_df.loc[i, "sentiment"] = classify_sentiment(news_df.loc[i, "title"])
    if (n + 1) % 10 == 0 or n == len(todo_idx) - 1:
        print(f"Classified {n + 1}/{len(todo_idx)} new headlines")
    time.sleep(0.5)  # be polite to the rate limit

news_df[["timestamp", "title", "sentiment"]].tail(10)

In [ ]:
news_df.to_csv("../data/news/ASII/asii_news_sentiment.csv", index=False)
print("Saved labeled headlines to ../data/news/ASII/asii_news_sentiment.csv")

# Step 3: Quick Summary

In [ ]:
counts = news_df["sentiment"].value_counts()

# Status colors, reserved for exactly this meaning -- not reused elsewhere,
# and never the only way the categories are told apart (the x-axis already
# labels each bar by name).
status_colors = {"positive": "#2ca02c", "negative": "#d62728", "unclear": "#7f7f7f", "error": "#bbbbbb"}
bar_colors = [status_colors.get(label, "#1f77b4") for label in counts.index]

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(counts.index, counts.values, color=bar_colors)
for i, v in enumerate(counts.values):
    ax.text(i, v + 0.2, str(v), ha="center")
ax.set_title(f"ASII headline sentiment ({len(news_df)} articles)")
ax.set_ylabel("Number of headlines")
plt.show()

print(counts)

## Caveats worth keeping in mind

- **LLM labels are opinions, not ground truth.** Spot-check a sample of headlines against their labels yourself before trusting them for anything downstream.
- **Small samples make the split noisy.** A 3-2 positive/negative split on 5 headlines says almost nothing; treat this like the RMSE baselines in the prediction notebooks -- don't over-read small numbers.
- **Keyword search casts a wider net than it filters.** "Astra International" can match unrelated mentions (e.g. other Astra Group subsidiaries quoted alongside it, or ASII covered only as a shareholder in a subsidiary's own news); skim `news_df['title']` for relevance before trusting the whole set.
- **This costs real money/credits per run.** Each OpenAI call and each Sectors news request has a cost -- avoid re-running the classification loop unnecessarily once you have a saved CSV.
- **Natural next step:** aggregate this into a daily sentiment feature (e.g. net positive-minus-negative headline count per day) the same way `UNTR Sentiment Window Model.ipynb` does for UNTR, then test it against the baseline the same honest way, instead of assuming it will help -- ASII being the group holding company likely means more news volume, which is exactly the coverage problem that notebook exists to check for.